# Hotel Booking Demand Analysis
## Feature Engineering
* **Goal:** Filling missing values and preparing Dataset for ML

In [ ]:
# Import libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

file_path = 'hotel_bookings.csv'

# Safe data loading with try/except
try:
    df = pd.read_csv(file_path)
    print('Load successfully')
except FileNotFoundError:
    print(f'File not found, check {file_path}')

In [ ]:
# First inspection of the dataset
df.head()

In [ ]:
# Statistic metrics of dataset
df.describe()

In [ ]:
# Checking the shape
print(f'Rows: {df.shape[0]} -- Columns: {df.shape[1]}')

In [ ]:
# Checking on missing values
df.isna().sum()

In [ ]:
print(df['children'].isna())

In [ ]:
# Checking the missing values in the column child - which hotel 
print(df[df['children'].isna()])

**Note on Missing Values (Children):**

The analysis revealed 4 missing records in the `children` column, all originating from the City Hotel segment. To handle these missing values without distorting the distribution, the mode (most frequent value) for the `children` column within the City Hotel segment will be calculated and utilized to impute the empty entries.

In [ ]:
# Creating variable for city hotel and children mode
mode_child_city = df[df['hotel'] == 'City Hotel']['children'].mode()[0]

# Filling the missing values with the 
df['children'] = df['children'].fillna(mode_child_city)

# Checking on missing values again
print(df.isna().sum())

In [ ]:
# Checking the unique values for column 'agent'
print(df['agent'].unique())

In [ ]:
# Checking the unique values for column 'company'
print(df['company'].unique())

In [ ]:
# Checking if 0 represents an agent or company as well
print(df[df['company']<1])
print(df[df['agent']<1])

**Note:** The missing values in both columns 'agent' and 'company' are not strings, they are numbers, which indicates the number represents a special agent in both hotels starting with 1. If I am using unknown, the columns will be converted into strings. Therefore I decided to fill the missing values with a 0. After checking both columns for the unique values it shows, that only integers in the columns

In [ ]:
# Missing values in `agent` and `company` indicate that no agent or company was involved. Since valid IDs start at 1 and no existing values are below 1, missing values are filled with 0.
df['agent'] = df['agent'].fillna(0)
df['company'] = df['company'].fillna(0)

# Check for missing values
print(df.isna().sum())


In [ ]:
# Filling missing 'country' values will UNK for unknown
df['country'] = df['country'].fillna('UNK')

# Checking for missing values
print(df.isna().sum())

**Note:** All missing values across the dataset have been successfully handled and imputed. The data contains no further null values and is ready for the next feature engineering steps.

In [ ]:
# Checking the dtype for reservation status type
df['reservation_status_date'].info()

In [ ]:
# Convert the reservation status date to datetime format
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'])

# Check the dtype after converting
df['reservation_status_date'].info()

In [ ]:
# Adding new column 'total_stay_length' by accumulate 'stays_in_weekend_nights' and 'stays_in_week_nights' for a further calculation of revenue 
df['total_stay_length'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

# Checking for unqiue values
print(df['total_stay_length'].unique())  

In [ ]:
# Adding new column 'revenue' multiply 'total_stay_length' with 'adr'
df['booking_value'] = df['total_stay_length'] * df['adr']


In [ ]:
# Calculate and prepare monthly booking value of confirmed bookings for visualization
confirmed_bookings = df[df['is_canceled'] == 0]
rev_by_month_hotel = confirmed_bookings.groupby([confirmed_bookings['reservation_status_date'].dt.to_period('M'), 'hotel'])['booking_value'].sum().reset_index()
rev_by_month_hotel['reservation_status_date'] = rev_by_month_hotel['reservation_status_date'].astype(str)

plt.figure(figsize=(16,6))
sns.lineplot(data=rev_by_month_hotel, x='reservation_status_date', y='booking_value', hue='hotel', marker='o')

plt.title('Monthly Revenue of Confirmed Bookings', fontsize=14, loc='left')
plt.xlabel('')
plt.xticks(rotation=70)
plt.ylabel('Revenue ($)', fontsize=12)
plt.ticklabel_format(style='plain', axis='y')
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left', title='Hotel')

plt.tight_layout()
plt.show()

### Analysis of Monthly Revenue for Confirmed Bookings

The chart reveals that the high seasons primarily occur during the summer months, aligning with the peak summer holiday period. Conversely, the low seasons begin in September and last until January, spanning the autumn and post-holiday winter weeks. To optimize occupancy and revenue during these quieter periods, hotel management should consider implementing targeted seasonal discounts or promotional campaigns.

In [ ]:
# Cross-checking deposit type with reservation status to confirm EDA cancellation patterns
canceled_df = df[df['is_canceled'] == 1]
canceled_df[['reservation_status', 'deposit_type']].value_counts()

### Analysis of Cross-Checking Deposit Type with Reservation Status

* The majority of canceled bookings did not require a deposit ("No Deposit" type). However, a significant portion of cancellations occurred under "Non Refund" rates, confirming the patterns analyzed in the previous EDA. Operatively, the dataset provides no clear indicator of whether these non-refundable bookings were actually refunded or if the hotels successfully collected the cancellation/no-show fees.

In [ ]:
# Adding new column 'TotalPax' for preparing for ML
df['total_pax'] = df['adults'] + df['children'] + df['babies']

# Checking for values
print(df['total_pax'].value_counts())


**Note on Total Pax (Total Guests):**

The analysis of the engineered `total_pax` feature reveals that the vast majority of bookings consist of 1 to 3 guests. Unusually high `total_pax` values strongly indicate potential group or corporate bookings. This consolidated feature will provide the machine learning model with a clearer indicator of party size compared to individual guest categories.

In [ ]:
# Checking the columns
df.columns

## Preparing Dataset for ML Model

In [ ]:
# Drop 'reservation_status' to prevent data leakage
df.drop(columns=['reservation_status'], inplace=True)

**Note on Data Leakage Prevention:**

Since the primary goal of this machine learning model is to forecast booking cancellations, `is_canceled` serves as our target variable. The `reservation_status` column contains post-booking outcomes such as 'Canceled' and 'No-Show'. Including this feature would cause severe data leakage, as it effectively reveals the target value beforehand—information that would not be available at the time of making a prediction. Therefore, this column must be dropped prior to model training.

In [ ]:
# Convert agent and company IDs into binary indicator features
df['has_company'] = (df['company'] > 0).astype(int)
df['has_agent'] = (df['agent'] > 0).astype(int)

# Check if it works
print(df[['has_agent', 'has_company']].head())

**Note:** The original `agent` and `company` columns contain ID values without ordinal meaning. Therefore, they were transformed into binary features indicating whether an agent or company was involved in the booking.

In [ ]:
# Drop 'agent' and 'company'
df.drop(columns=['agent', 'company'], inplace=True)

# Check if it works
df.columns